# Detection tuning view — Plot 1

Tune knock detection on a recording (real from the ESP32-S2, or synthetic from `sim.py`). Edit the parameters in the next cell, re-run all cells, and watch onsets appear, disappear, or move.

**How the detector works, per channel**

1. **Band-pass** the raw ADC signal to `BAND_HZ` (zero-phase, so filtering does not shift times).
2. **Envelope**: loudness of the filtered signal over time.
3. **Noise floor**: median envelope over the past `NOISE_WIN_S` seconds.
4. **Trigger** when the envelope rises above `K_RATIO × noise floor`.
5. **Walk back** from the trigger to where the envelope crossed `floor + LOW_FRAC × (peak − floor)`, and **interpolate between samples** there. That point is the onset written to `detections.csv`.

**The three parameters that matter**

| Parameter | What it changes | Too low | Too high |
|---|---|---|---|
| `K_RATIO` | How loud a sound must be to count | False onsets on noise bumps (orange "false" rows) | Quiet or distant knocks missed |
| `LOW_FRAC` | Where on the rising edge the onset is placed | Onset lands in the noise, so it jitters | Onset lands late on the edge; more sensitive to loudness differences between sensors |
| `BAND_HZ` | Which frequencies count | Low edge too low: hum and footsteps pass | Band too narrow or high: knock energy cut, onsets late or missed |

**What good tuning looks like**
- Every knock you made shows a blue onset line on **every** channel, with no blue lines where you didn't knock.
- In the zoom panel, the onset dot sits at the **foot of the rising edge**, not in the noise and not halfway up.
- The **time-gap error** (difference between channels) is small and stable. A constant early or late offset on all channels does not matter, because it cancels in the gap. Localization only uses gaps.


In [ ]:
# ===================== EDIT THESE =====================
RUN = "sim-direction"            # folder under data/runs/ (None for a bare recording)
RECORDING = None                 # None = take it from the run's run.yaml; or e.g. "data/recordings/stage1.npz"
ARRAY = None                     # None = from run.yaml; or "config/array.yaml"

K_RATIO  = 8.0                   # trigger = envelope above K_RATIO x noise floor
LOW_FRAC = 0.2                   # onset = where envelope crosses floor + LOW_FRAC x (peak - floor)
BAND_HZ  = (300.0, 3000.0)       # band-pass edges (Hz); top is capped at 0.45 x fs

# second-tier knobs (usually leave alone)
ENV_SMOOTH_MS = 1.0
NOISE_WIN_S   = 2.0

ZOOM_SOUND = None                # sound_id to zoom on; None = loudest onset on ZOOM_CH
ZOOM_CH    = 0
SAVE_DETECTIONS = False          # True = write this tuning's onsets to data/runs/<RUN>/detections.csv
# ======================================================

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, yaml
import matplotlib.pyplot as plt
import detect as D
from arraycfg import load_array

run_dir = ROOT / "data/runs" / RUN if RUN else None
meta = yaml.safe_load((run_dir / "run.yaml").read_text()) if run_dir and (run_dir / "run.yaml").exists() else {}
rec_path = RECORDING or meta.get("recording")
cfg = load_array(ARRAY or meta.get("array", "config/array.yaml"))
rec = np.load(rec_path)
samples, fs = rec["samples"], int(rec["fs_hz"])
if samples.ndim == 1:
    samples = samples[:, None]
names = cfg["_names"][: samples.shape[1]] if len(cfg["_names"]) >= samples.shape[1] else [f"ch{c}" for c in range(samples.shape[1])]
ref = pd.read_csv(run_dir / "detections.csv") if run_dir and (run_dir / "detections.csv").exists() else None
truth = D.true_arrivals(run_dir) if run_dir else None
print(f"{rec_path}: {samples.shape[0]/fs:.1f} s, {samples.shape[1]} channel(s) at {fs} Hz; c = {cfg['_c_mps']:.1f} m/s")
print("reference onsets:", "detections.csv" if ref is not None else "none", "| ground truth:", "yes (sim)" if truth is not None else "no")

In [ ]:
p = D.DetectParams(k_ratio=K_RATIO, low_frac=LOW_FRAC, band_hz=BAND_HZ,
                   env_smooth_ms=ENV_SMOOTH_MS, noise_win_s=NOISE_WIN_S)
det, chains, _ = D.detect_recording(samples, fs, p, cfg["_xy"][: samples.shape[1]], cfg["_c_mps"])

# Judge against ground truth if this is a sim; otherwise against detections.csv; otherwise just count.
if truth is not None:
    cmp = D.compare(det, truth.rename(columns={"t_true": "t_ref"})); judged = "truth"
elif ref is not None:
    cmp = D.compare(det, ref.rename(columns={"t_s": "t_ref"})); judged = "detections.csv"
else:
    cmp, judged = None, None
summary = D.summarize(cmp) if cmp is not None else None
title = D.finding_title(det, summary, names, truth_based=(judged == "truth"))
print(title)
if summary:
    print(f"implied per-sensor timing jitter: {summary['sigma_t_implied_s']*1e6:.0f} us "
          f"(array file assumes {cfg['timing']['sigma_s']*1e6:.0f} us)")
    display(cmp[cmp.status != "found"]) if (cmp.status != "found").any() else print("no missed or false onsets")

In [ ]:
fig = D.plot_tuning(samples, fs, chains, det, p, ref=ref, truth=truth, names=names,
                    zoom_sound=ZOOM_SOUND, zoom_ch=ZOOM_CH, title=title)
out = ROOT / "docs/figures" / f"plot1_detection_tuning_{RUN or Path(rec_path).stem}.png"
fig.savefig(out, dpi=130)
print("saved", out.relative_to(ROOT))
plt.show()

### How to read it: early, late, or missed?

- **Missed:** a knock is visible in the raw or filtered trace but has **no blue line**, or the line is on some channels only. Lower `K_RATIO`, or check that `BAND_HZ` covers the knock (look at the filtered trace: if the knock almost vanishes there, the band is wrong).
- **False:** a **blue line with no knock** under it, usually on a noise bump that just crossed the red threshold. Raise `K_RATIO`.
- **Early:** in the zoom panel the blue onset dot sits **before the envelope leaves the noise floor**. Raise `LOW_FRAC`.
- **Late:** the dot sits **well up the rising edge**, after the dotted true arrival (sims) or the gray dashed reference. Lower `LOW_FRAC`.
- **The number that decides it:** *time-gap error* in the title: the typical error in the time differences between channels, which is all localization uses. The notebook converts it to a per-sensor jitter (robust spread ÷ √2): compare that to `timing.sigma_s` in `config/array.yaml`. *Outliers* (> 0.5 ms) are counted separately; they are almost always soft voice onsets.
- **Voices** start softly, so their onset is 10-20 ms late, and on some channels the walk-back lands on a different syllable edge, giving a gap error of several ms. Tune on **knocks**. Voices will need cross-correlation between channels (comparing whole waveforms) instead of onset picking; that is a later step.


### Sweep: what each parameter does here
Same recording, one parameter varied at a time, everything else as set above.

In [ ]:
def run_with(**kw):
    q = D.DetectParams(**{**dict(k_ratio=K_RATIO, low_frac=LOW_FRAC, band_hz=BAND_HZ,
                                 env_smooth_ms=ENV_SMOOTH_MS, noise_win_s=NOISE_WIN_S), **kw})
    d, _, _ = D.detect_recording(samples, fs, q, cfg["_xy"][: samples.shape[1]], cfg["_c_mps"])
    if cmp is None:
        return dict(onsets=len(d), sounds=d.sound_id.nunique() if len(d) else 0)
    r = truth.rename(columns={"t_true": "t_ref"}) if judged == "truth" else ref.rename(columns={"t_s": "t_ref"})
    s = D.summarize(D.compare(d, r))
    return dict(found=s["found"], missed=s["missed"], false=s["false"], gap_err_us=round(s["gap_typical_us"], 1), outliers=s["gap_outliers"])

rows = [dict(param="k_ratio", value=v, **run_with(k_ratio=v)) for v in (2, 4, 8, 16, 32, 64)]
rows += [dict(param="low_frac", value=v, **run_with(low_frac=v)) for v in (0.05, 0.1, 0.2, 0.3, 0.5, 0.8)]
rows += [dict(param="band_hz", value=str(v), **run_with(band_hz=v)) for v in ((50, 4000), (300, 3000), (800, 3000), (1500, 4000))]
pd.DataFrame(rows)

### Optional: live sliders
Needs `pip install ipywidgets`. Moves the parameters and redraws the figure.

In [ ]:
try:
    import ipywidgets as w
    def _redraw(k_ratio, low_frac, band_lo, band_hi):
        q = D.DetectParams(k_ratio=k_ratio, low_frac=low_frac, band_hz=(band_lo, band_hi))
        d, ch, _ = D.detect_recording(samples, fs, q, cfg["_xy"][: samples.shape[1]], cfg["_c_mps"])
        r = truth.rename(columns={"t_true": "t_ref"}) if judged == "truth" else (ref.rename(columns={"t_s": "t_ref"}) if ref is not None else None)
        s = D.summarize(D.compare(d, r)) if r is not None else None
        D.plot_tuning(samples, fs, ch, d, q, ref=ref, truth=truth, names=names, zoom_sound=ZOOM_SOUND,
                      zoom_ch=ZOOM_CH, title=D.finding_title(d, s, names, judged == "truth"))
        plt.show()
    w.interact_manual(_redraw, k_ratio=w.FloatLogSlider(value=K_RATIO, base=2, min=0, max=7, step=0.25),
                      low_frac=w.FloatSlider(value=LOW_FRAC, min=0.02, max=0.9, step=0.02),
                      band_lo=w.FloatSlider(value=BAND_HZ[0], min=20, max=2000, step=20),
                      band_hi=w.FloatSlider(value=BAND_HZ[1], min=500, max=0.45 * fs, step=50))
except ImportError:
    print("ipywidgets not installed: edit the parameter cell and re-run instead.")

### Save these onsets
For a real recording with no `detections.csv` yet (Stage 1 and Stage 2 stream raw samples to the laptop, so this detector *is* the detector). Set `SAVE_DETECTIONS = True` at the top. Sim runs keep their reference file unless you do.

In [ ]:
if SAVE_DETECTIONS:
    target = (run_dir or ROOT / "data/runs" / Path(rec_path).stem)
    target.mkdir(parents=True, exist_ok=True)
    D.to_contract(det).to_csv(target / "detections.csv", index=False)
    (target / "detect_params.yaml").write_text(yaml.safe_dump({k: list(v) if isinstance(v, tuple) else v
                                                              for k, v in D.asdict(p).items()}))
    print("wrote", (target / "detections.csv").relative_to(ROOT), "and detect_params.yaml")
else:
    print("not saved (SAVE_DETECTIONS = False)")

### Stage 2: measure the real jitter
Knock **20 times from one taped spot**, record, set `RUN`/`RECORDING` to that recording, and run this cell. Per-sensor jitter = spread of the time gap ÷ √2. Put the result in `timing.sigma_s` in `config/array.yaml` and rerun the M0 panel B.

In [ ]:
if det.sound_id.nunique() >= 2 and samples.shape[1] >= 2:
    j = D.measure_jitter(det[det.groupby("sound_id").ch.transform("nunique") >= 2], 0, 1)
    print(f"{j['n']} sounds heard on both {names[0]} and {names[1]}: gap mean {j['gap_mean_us']:.0f} us, "
          f"spread {j['gap_std_us']:.0f} us  ->  timing.sigma_s = {j['sigma_s']:.6f}")
    print("Only meaningful when every sound came from the SAME spot (on a sim demo run it mixes spots).")
else:
    print("Need >= 2 channels and >= 2 sounds.")